# COVID-19 US Counties ETL Pipeline – Medallion Architecture in Databricks

**Objective**: Demonstrate end-to-end data engineering best practices:  
- Ingest raw data (Bronze)  
- Clean + validate + enrich (Silver)  
- Aggregate + business-ready insights (Gold)  
- Use Delta Lake for reliability (ACID, schema enforcement, time travel)  
- Schedule as production job  


**Layers**:
- **Bronze**: Raw, immutable ingestion (as-is data)
- **Silver**: Cleaned, validated, enriched (ready for analysis)
- **Gold**: Aggregated, business insights (reporting-ready)

Dataset: Databricks sample – `/databricks-datasets/COVID/covid-19-data/us-counties.csv` (columns: date, county, state, fips, cases, deaths)

In [0]:
# List available datasets
display(dbutils.fs.ls('/databricks-datasets'))

## Bronze Layer: Raw Data Ingestion
- Load as-is (no transformation)
- Save as Delta table (immutable, versioned)
- Purpose: Audit trail, reprocess if needed

In [0]:
bronze_df=spark.read.csv('dbfs:/databricks-datasets/COVID/covid-19-data/us-counties.csv',
                         header=True, inferSchema=True)

display(bronze_df)
print(f"Bronze df count {bronze_df.count()}")

bronze_df.write.format("delta").saveAsTable("covid_bronze")


In [0]:
display(spark.sql("SELECT * FROM covid_bronze LIMIT 10"))

In [0]:
from pyspark.sql.functions import col, sum as sum_

# Count nulls in every column of the bronze table
null_counts = bronze_df.select([
    sum_(col(c).isNull().cast("int")).alias(c) 
    for c in bronze_df.columns
])

display(null_counts)

In [0]:
display(bronze_df.select(
    sum_(col("fips").isNull().cast("int")).alias("fips_null_count"),
    sum_(col("deaths").isNull().cast("int")).alias("deaths_null_count")
))